# Oracle Routing Potential Test

This notebook checks whether routing between already-trained forecasting experts can theoretically improve performance before training a router. It uses the chronological `router_train` split only and evaluates a perfect window-level oracle that always picks the lowest-error expert for each complete forecasting window.

## 1. Setup

Imports project helpers, selects the existing CUDA/CPU device, and defines the canonical forecasting dimensions used by the trained ETTh1 checkpoints.

In [ ]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("OMP_NUM_THREADS", "1")

from dataclasses import fields, is_dataclass
from importlib import import_module
from itertools import combinations
from pathlib import Path
import gc
import math
import sys
import traceback

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

try:
    from IPython.display import display
except Exception:
    def display(value):
        print(value)

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from basicts.scaler import ZScoreScaler
from scripts.chronological_expert_training import (
    DEFAULT_INPUT_LEN,
    DEFAULT_NUM_FEATURES,
    DEFAULT_OUTPUT_LEN,
    _assert_full_data_contract,
    _assert_no_expert_gradients,
    _load_torch_checkpoint,
    _prepare_forecasting_batch,
    assert_experts_frozen,
    load_full_chronological_data,
    prepare_chronological_dataloaders,
)

DATA_DIR = ROOT / "datasets" / "ETTh1"
CHECKPOINT_DIR = ROOT / "checkpoints"
BATCH_SIZE = 512
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 7
TIE_TOLERANCE = 1e-12

print(f"Project root: {ROOT}")
print(f"Device: {DEVICE}")
print(f"Input/output/features: {DEFAULT_INPUT_LEN}/{DEFAULT_OUTPUT_LEN}/{DEFAULT_NUM_FEATURES}")

## 2. Load Chronological Router-Train Data

The scaler is fit only on `expert_train` by the existing project helper, then reused for the `router_train` split. The `router_train` dataloader is asserted to be non-shuffled.

In [ ]:
np.random.seed(SEED)
torch.manual_seed(SEED)

full_data = load_full_chronological_data(DATA_DIR)
_assert_full_data_contract(full_data, DEFAULT_NUM_FEATURES)

loaders, scaler = prepare_chronological_dataloaders(
    full_data=full_data,
    scaler=ZScoreScaler(norm_each_channel=True, rescale=False),
    batch_size=BATCH_SIZE,
    input_len=DEFAULT_INPUT_LEN,
    output_len=DEFAULT_OUTPUT_LEN,
)

router_train_loader = loaders["router_train"]
assert getattr(router_train_loader.dataset, "split_role", None) == "router_train"
assert isinstance(router_train_loader.sampler, torch.utils.data.SequentialSampler)

number_of_windows = len(router_train_loader.dataset)
print(f"Full data shape: {full_data.shape}")
print(f"Router-train windows: {number_of_windows}")
print("Router-train dataloader shuffle=False confirmed")

## 3. Discover Trained Forecasting Experts

This section finds trained expert checkpoints already present in the repository. It prioritizes the router-pipeline checkpoints and also includes benchmark forecasting checkpoints when their model class can be reconstructed from the existing project definitions. Each candidate is validated later before it participates in the oracle analysis.

In [ ]:
def _torch_load(path):
    return _load_torch_checkpoint(path, torch.device("cpu"))


def _model_class_from_module(module, module_name, preferred_class=None):
    if preferred_class and hasattr(module, preferred_class):
        return getattr(module, preferred_class)
    forecasting_names = [name for name in dir(module) if "Forecast" in name]
    if forecasting_names:
        return getattr(module, sorted(forecasting_names)[0])
    if hasattr(module, module_name):
        return getattr(module, module_name)
    return None


def _config_class_from_module(module, module_name):
    preferred_name = f"{module_name}Config"
    if hasattr(module, preferred_name):
        return getattr(module, preferred_name)
    for name in dir(module):
        if name.endswith("Config"):
            return getattr(module, name)
    return None


def _config_kwargs(config_class, values):
    if not is_dataclass(config_class):
        return values
    valid_names = {field.name for field in fields(config_class)}
    return {name: value for name, value in values.items() if name in valid_names}


def _default_config_values():
    return {
        "input_len": DEFAULT_INPUT_LEN,
        "output_len": DEFAULT_OUTPUT_LEN,
        "label_len": DEFAULT_INPUT_LEN // 2,
        "num_features": DEFAULT_NUM_FEATURES,
        "num_classes": None,
    }


def _build_config(config_class, checkpoint, extra_config=None):
    values = _default_config_values()
    if isinstance(checkpoint.get("model_config"), dict):
        values.update(checkpoint["model_config"])
    values.update(extra_config or {})
    return config_class(**_config_kwargs(config_class, values))


def _checkpoint_candidates():
    candidate_specs = [
        {
            "expert_name": "Candidate_DLinear",
            "module_name": "DLinear",
            "model_class_name": "DLinear",
            "checkpoint_path": CHECKPOINT_DIR / "candidates" / "best_dlinear.pt",
        },
        {
            "expert_name": "Candidate_PatchTST",
            "module_name": "PatchTST",
            "model_class_name": "PatchTSTForForecasting",
            "checkpoint_path": CHECKPOINT_DIR / "candidates" / "best_patchtst.pt",
        },
        {
            "expert_name": "Candidate_iTransformer",
            "module_name": "iTransformer",
            "model_class_name": "iTransformerForForecasting",
            "checkpoint_path": CHECKPOINT_DIR / "candidates" / "best_itransformer.pt",
        },
        {
            "expert_name": "Candidate_TimesNet",
            "module_name": "TimesNet",
            "model_class_name": "TimesNetForForecasting",
            "checkpoint_path": CHECKPOINT_DIR / "candidates" / "best_timesnet.pt",
        },
        {
            "expert_name": "Candidate_ModernTCN",
            "module_name": "ModernTCN",
            "model_class_name": "ModernTCNForForecasting",
            "checkpoint_path": CHECKPOINT_DIR / "candidates" / "best_moderntcn.pt",
        },
    ]
    return [candidate for candidate in candidate_specs if candidate["checkpoint_path"].exists()]


candidate_experts = _checkpoint_candidates()
candidate_experts_df = pd.DataFrame(
    [{**candidate, "checkpoint_path": str(candidate["checkpoint_path"])} for candidate in candidate_experts]
)
print(f"Discovered {len(candidate_experts)} candidate expert checkpoints")
display(candidate_experts_df)

## Candidate Checkpoint Summary

This display-only section shows which candidate checkpoints exist and the validation metadata saved inside them. It does not train models or change checkpoint files.

In [ ]:
def _checkpoint_metric(checkpoint, *names):
    for name in names:
        value = checkpoint.get(name)
        if value is not None:
            return value
    return None


checkpoint_summary_rows = []
for candidate in candidate_experts:
    checkpoint = _torch_load(candidate["checkpoint_path"])
    checkpoint_summary_rows.append(
        {
            "expert_name": checkpoint.get("expert_name", candidate["expert_name"]),
            "model_key": checkpoint.get("model_key"),
            "checkpoint": candidate["checkpoint_path"].name,
            "checkpoint_path": str(candidate["checkpoint_path"]),
            "best_epoch": _checkpoint_metric(checkpoint, "best_epoch", "epoch"),
            "best_validation_mae": _checkpoint_metric(
                checkpoint,
                "best_validation_mae",
                "validation_mae",
                "val_mae",
            ),
            "input_len": _checkpoint_metric(checkpoint, "input_len") or checkpoint.get("model_config", {}).get("input_len"),
            "forecast_len": _checkpoint_metric(checkpoint, "forecast_len", "output_len") or checkpoint.get("model_config", {}).get("output_len"),
            "num_features": _checkpoint_metric(checkpoint, "num_features") or checkpoint.get("model_config", {}).get("num_features"),
        }
    )

candidate_checkpoint_summary = pd.DataFrame(checkpoint_summary_rows)
if not candidate_checkpoint_summary.empty:
    candidate_checkpoint_summary = candidate_checkpoint_summary.sort_values(
        ["best_validation_mae", "expert_name"],
        na_position="last",
        ignore_index=True,
    )

print("Saved candidate checkpoint results")
display(candidate_checkpoint_summary)

print("Training commands to run in a training notebook or terminal, not here:")
print("  python scripts/train_candidate_experts.py --models moderntcn")
print("  python scripts/train_candidate_experts.py --models all")

## 4. Load Each Expert Once And Store Per-Window Losses

Each compatible expert is loaded from its checkpoint, placed in evaluation mode, frozen, evaluated on `router_train` under `torch.no_grad()`, and then discarded. This avoids duplicate inference: every expert's per-window losses are generated once and reused for every later combination.

In [ ]:
def _prediction_tensor(output):
    if isinstance(output, dict):
        if "prediction" not in output:
            raise KeyError("Model output dictionary does not contain 'prediction'")
        return output["prediction"]
    return output


def _call_model(model, inputs, targets):
    try:
        output = model(inputs)
    except TypeError:
        try:
            output = model(inputs, None)
        except TypeError:
            output = model(inputs, targets)
    return _prediction_tensor(output)


def _per_window_mae(prediction, target, target_mask):
    assert prediction.shape == target.shape, (
        f"Prediction shape {tuple(prediction.shape)} does not match target shape {tuple(target.shape)}"
    )
    mask = target_mask.to(prediction.dtype)
    valid_counts = mask.sum(dim=(1, 2))
    assert torch.all(valid_counts > 0), "Every window must contain at least one valid target value"
    absolute_error = torch.abs(prediction - target) * mask
    return absolute_error.sum(dim=(1, 2)) / valid_counts


def _load_candidate_model(candidate):
    checkpoint = _torch_load(candidate["checkpoint_path"])
    module = import_module(f"basicts.models.{candidate['module_name']}")
    model_class = _model_class_from_module(module, candidate["module_name"], candidate.get("model_class_name"))
    config_class = _config_class_from_module(module, candidate["module_name"])
    if model_class is None or config_class is None:
        raise RuntimeError("Could not locate model/config class")
    config = _build_config(config_class, checkpoint)
    model = model_class(config)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.to(DEVICE)
    model.eval()
    model.requires_grad_(False)
    assert_experts_frozen(model)
    return model, checkpoint


def evaluate_candidate_on_router_train(candidate):
    model, checkpoint = _load_candidate_model(candidate)
    states_before = {name: value.detach().clone() for name, value in model.state_dict().items()}
    losses = []
    with torch.no_grad():
        for batch in router_train_loader:
            inputs, targets, targets_mask = _prepare_forecasting_batch(batch, DEVICE, scaler)
            prediction = _call_model(model, inputs, targets).detach()
            assert prediction.shape == targets.shape, (
                f"{candidate['expert_name']} prediction shape {tuple(prediction.shape)} does not match target shape {tuple(targets.shape)}"
            )
            losses.extend(_per_window_mae(prediction, targets, targets_mask).cpu().tolist())
    _assert_no_expert_gradients(model)
    for parameter_name, value in model.state_dict().items():
        assert torch.equal(value, states_before[parameter_name]), (
            f"{candidate['expert_name']} parameter changed during analysis: {parameter_name}"
        )
    del model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    gc.collect()
    return np.array(losses, dtype=np.float64), checkpoint


expert_loss_values = {}
loaded_expert_metadata = []
skipped_experts = []

for index, candidate in enumerate(candidate_experts, start=1):
    print(f"[{index}/{len(candidate_experts)}] Evaluating {candidate['expert_name']} from {candidate['checkpoint_path']}")
    try:
        losses, checkpoint = evaluate_candidate_on_router_train(candidate)
        assert len(losses) == number_of_windows
        expert_name = candidate["expert_name"]
        if expert_name in expert_loss_values:
            suffix = 2
            while f"{expert_name}_{suffix}" in expert_loss_values:
                suffix += 1
            expert_name = f"{expert_name}_{suffix}"
        expert_loss_values[expert_name] = losses
        loaded_expert_metadata.append(
            {
                "expert_name": expert_name,
                "checkpoint_path": str(candidate["checkpoint_path"]),
                "module_name": candidate["module_name"],
                "model_class_name": candidate.get("model_class_name"),
                "checkpoint_epoch": checkpoint.get("epoch"),
                "checkpoint_validation_mae": checkpoint.get("validation_mae", checkpoint.get("val_mae")),
                "router_train_mae": float(losses.mean()),
            }
        )
        print(f"  OK router_train MAE={losses.mean():.6f}")
    except Exception as exc:
        skipped_experts.append(
            {
                "expert_name": candidate["expert_name"],
                "checkpoint_path": str(candidate["checkpoint_path"]),
                "error": f"{type(exc).__name__}: {exc}",
                "traceback": traceback.format_exc(limit=4),
            }
        )
        print(f"  SKIPPED: {type(exc).__name__}: {exc}")

loaded_experts_df = pd.DataFrame(loaded_expert_metadata).sort_values("router_train_mae", ignore_index=True)
skipped_experts_df = pd.DataFrame(skipped_experts)
expert_names = list(expert_loss_values.keys())
assert len(expert_names) >= 2, "Need at least two compatible trained experts for routing analysis"

print(f"\nLoaded {len(expert_names)} compatible experts")
display(loaded_experts_df)
if not skipped_experts_df.empty:
    print("Skipped incompatible candidates")
    display(skipped_experts_df[["expert_name", "checkpoint_path", "error"]])

## 5. Single-Combination Diagnostic

This preserves the original diagnostic view: pick one combination of at least two experts, calculate each expert's overall MAE, the best single expert, the per-window oracle MAE, wins, ties, and routing-potential interpretation.

In [ ]:
def routing_potential_label(percentage_improvement):
    if percentage_improvement < 1.0:
        return "experts are not complementary enough; router unlikely to meaningfully improve"
    if percentage_improvement <= 3.0:
        return "limited routing potential"
    return "meaningful routing potential exists"


def analyze_combination(combination):
    combination = tuple(combination)
    matrix = np.column_stack([expert_loss_values[name] for name in combination])
    oracle_losses = matrix.min(axis=1)
    tied_mask = np.isclose(matrix, oracle_losses[:, None], atol=TIE_TOLERANCE, rtol=0.0).sum(axis=1) > 1
    winner_indices = matrix.argmin(axis=1)
    winning_experts = ["Tie" if tied else combination[index] for tied, index in zip(tied_mask, winner_indices)]
    per_window = pd.DataFrame({"window_index": np.arange(number_of_windows)})
    for index, expert_name in enumerate(combination):
        per_window[f"{expert_name}_mae"] = matrix[:, index]
    per_window["oracle_mae"] = oracle_losses
    per_window["winning_expert"] = winning_experts
    per_window["tied"] = tied_mask

    expert_mae = {expert_name: float(expert_loss_values[expert_name].mean()) for expert_name in combination}
    best_single_expert = min(expert_mae, key=expert_mae.get)
    best_single_mae = expert_mae[best_single_expert]
    oracle_mae = float(oracle_losses.mean())
    absolute_improvement = best_single_mae - oracle_mae
    percentage_improvement = (absolute_improvement / best_single_mae * 100.0) if best_single_mae else 0.0
    tied_windows = int(tied_mask.sum())
    win_counts = {
        expert_name: int(((per_window["winning_expert"] == expert_name) & ~per_window["tied"]).sum())
        for expert_name in combination
    }
    expert_results_local = pd.DataFrame(
        [
            {
                "expert_name": expert_name,
                "mae": expert_mae[expert_name],
                "windows_won": win_counts[expert_name],
                "win_percentage": win_counts[expert_name] / number_of_windows * 100.0,
            }
            for expert_name in combination
        ]
    ).sort_values("mae", ignore_index=True)
    summary = {
        "expert_mae": expert_mae,
        "best_single_expert": best_single_expert,
        "best_single_mae": float(best_single_mae),
        "oracle_mae": float(oracle_mae),
        "absolute_improvement": float(absolute_improvement),
        "percentage_improvement": float(percentage_improvement),
        "routing_potential": routing_potential_label(percentage_improvement),
        "number_of_windows": int(number_of_windows),
        "tied_windows": int(tied_windows),
    }
    return expert_results_local, summary, per_window


def summarize_combination_fast(combination):
    combination = tuple(combination)
    matrix = np.column_stack([expert_loss_values[name] for name in combination])
    oracle_losses = matrix.min(axis=1)
    expert_mae = {expert_name: float(expert_loss_values[expert_name].mean()) for expert_name in combination}
    best_single_expert = min(expert_mae, key=expert_mae.get)
    best_single_mae = expert_mae[best_single_expert]
    oracle_mae = float(oracle_losses.mean())
    absolute_improvement = best_single_mae - oracle_mae
    percentage_improvement = (absolute_improvement / best_single_mae * 100.0) if best_single_mae else 0.0
    return {
        "combination": " + ".join(combination),
        "number_of_experts": len(combination),
        "best_single_expert": best_single_expert,
        "best_single_mae": float(best_single_mae),
        "oracle_mae": float(oracle_mae),
        "absolute_improvement": float(absolute_improvement),
        "percentage_improvement": float(percentage_improvement),
        "routing_potential": routing_potential_label(percentage_improvement),
    }


preferred_single_combination = [name for name in ["DLinear", "Transformer"] if name in expert_loss_values]
if len(preferred_single_combination) < 2:
    preferred_single_combination = expert_names[:2]

SINGLE_COMBINATION = tuple(preferred_single_combination)
expert_results, routing_summary, per_window_results = analyze_combination(SINGLE_COMBINATION)

print(f"Single-combination diagnostic: {SINGLE_COMBINATION}")
display(expert_results)
display(per_window_results.head())
routing_summary

## 6. Print Single-Combination Interpretation

In [ ]:
print("Overall expert MAE on router_train")
for expert_name, mae in routing_summary["expert_mae"].items():
    print(f"  {expert_name}: {mae:.6f}")

print("\nBest single expert")
print(f"  {routing_summary['best_single_expert']}: {routing_summary['best_single_mae']:.6f}")

print("\nOracle MAE")
print(f"  {routing_summary['oracle_mae']:.6f}")

print("\nOracle improvement over best single expert")
print(f"  Absolute: {routing_summary['absolute_improvement']:.6f}")
print(f"  Percentage: {routing_summary['percentage_improvement']:.2f}%")

patchtst_names = [name for name in SINGLE_COMBINATION if "patchtst" in name.lower()]
if patchtst_names:
    for patchtst_name in patchtst_names:
        patchtst_mae = routing_summary["expert_mae"][patchtst_name]
        patchtst_absolute = patchtst_mae - routing_summary["oracle_mae"]
        patchtst_percentage = patchtst_absolute / patchtst_mae * 100.0 if patchtst_mae else 0.0
        print(f"\nOracle improvement relative to {patchtst_name}")
        print(f"  Absolute: {patchtst_absolute:.6f}")
        print(f"  Percentage: {patchtst_percentage:.2f}%")
else:
    print("\nPatchTST is not in the selected single-combination diagnostic.")

print("\nWindow wins")
for _, row in expert_results.iterrows():
    print(f"  {row['expert_name']}: {int(row['windows_won'])} windows ({row['win_percentage']:.2f}%)")
print(f"  Ties: {routing_summary['tied_windows']} windows ({routing_summary['tied_windows'] / number_of_windows * 100.0:.2f}%)")

print("\nFinal interpretation")
if routing_summary["percentage_improvement"] < 1.0:
    print("  Below 1% oracle improvement: the experts are not complementary enough, so a router is unlikely to produce a meaningful improvement.")
elif routing_summary["percentage_improvement"] <= 3.0:
    print("  Between 1% and 3%: routing potential is limited.")
else:
    print("  Above 3%: meaningful routing potential exists.")

## 7. Single-Combination Difference Histogram

For two selected experts, this plots the distribution of per-window MAE differences.

In [ ]:
if len(SINGLE_COMBINATION) == 2:
    first_expert, second_expert = SINGLE_COMBINATION
    loss_difference = per_window_results[f"{first_expert}_mae"] - per_window_results[f"{second_expert}_mae"]
    absolute_loss_difference = loss_difference.abs()
    difference_stats = {
        "mean_difference": float(loss_difference.mean()),
        "median_difference": float(loss_difference.median()),
        "std_difference": float(loss_difference.std(ddof=0)),
        "mean_absolute_difference": float(absolute_loss_difference.mean()),
        "median_absolute_difference": float(absolute_loss_difference.median()),
        "std_absolute_difference": float(absolute_loss_difference.std(ddof=0)),
        "max_absolute_difference": float(absolute_loss_difference.max()),
    }
    print(f"Difference = {first_expert} MAE - {second_expert} MAE")
    for key, value in difference_stats.items():
        print(f"  {key}: {value:.6f}")
    plt.figure(figsize=(10, 5))
    plt.hist(loss_difference, bins=50, edgecolor="black", alpha=0.75)
    plt.axvline(0.0, color="black", linestyle="--", linewidth=1)
    plt.xlabel(f"{first_expert} window MAE - {second_expert} window MAE")
    plt.ylabel("Number of router-train windows")
    plt.title("Per-window expert MAE difference on router_train")
    plt.tight_layout()
    plt.show()
else:
    difference_stats = {}
    print("Histogram skipped because the selected diagnostic combination has more than two experts.")

## Test All Expert Combinations

This section reuses the stored per-window losses and evaluates every valid combination containing at least two experts: every pair, every group of three, and so on up to the group containing all available compatible experts. No additional model inference is performed here.

In [ ]:
combination_rows = []
for combination_size in range(2, len(expert_names) + 1):
    for combination in combinations(expert_names, combination_size):
        combination_rows.append(summarize_combination_fast(combination))

all_combinations_df = pd.DataFrame(combination_rows).sort_values(
    ["oracle_mae", "number_of_experts"],
    ascending=[True, True],
    ignore_index=True,
)

print(f"Evaluated {len(all_combinations_df)} expert combinations from {len(expert_names)} compatible experts")
display(all_combinations_df)

## Combination Highlights

This prints the lowest-oracle-MAE combination, the largest-percentage-improvement combination, the best two-expert combination, and whether larger expert groups meaningfully improve the oracle result.

In [ ]:
lowest_oracle_row = all_combinations_df.iloc[0]
largest_percentage_row = all_combinations_df.sort_values(
    ["percentage_improvement", "oracle_mae"],
    ascending=[False, True],
).iloc[0]
best_two_expert_row = all_combinations_df[all_combinations_df["number_of_experts"] == 2].iloc[0]

print("Combination with the lowest oracle MAE")
print(f"  {lowest_oracle_row['combination']}")
print(f"  Oracle MAE: {lowest_oracle_row['oracle_mae']:.6f}")

print("\nCombination with the largest percentage improvement")
print(f"  {largest_percentage_row['combination']}")
print(f"  Improvement: {largest_percentage_row['percentage_improvement']:.2f}%")

print("\nBest two-expert combination")
print(f"  {best_two_expert_row['combination']}")
print(f"  Oracle MAE: {best_two_expert_row['oracle_mae']:.6f}")

best_by_size = all_combinations_df.sort_values(
    ["number_of_experts", "oracle_mae"],
    ascending=[True, True],
).groupby("number_of_experts", as_index=False).first()
display(best_by_size[["number_of_experts", "combination", "oracle_mae", "percentage_improvement"]])
best_pair_oracle = float(best_by_size.loc[best_by_size["number_of_experts"] == 2, "oracle_mae"].iloc[0])
best_overall_oracle = float(lowest_oracle_row["oracle_mae"])
extra_expert_gain = best_pair_oracle - best_overall_oracle
extra_expert_gain_pct = extra_expert_gain / best_pair_oracle * 100.0 if best_pair_oracle else 0.0

print("\nDoes adding more experts meaningfully improve the oracle result?")
if extra_expert_gain_pct >= 1.0:
    print(f"  Yes. Best overall improves over the best pair by {extra_expert_gain:.6f} MAE ({extra_expert_gain_pct:.2f}%).")
else:
    print(f"  Not meaningfully. Best overall improves over the best pair by only {extra_expert_gain:.6f} MAE ({extra_expert_gain_pct:.2f}%).")

## Marginal Oracle Improvement By Combination Size

This compares the best oracle MAE available with two experts, then three experts, then four experts, continuing until all compatible experts are included.

In [ ]:
marginal_rows = []
previous_row = None
for _, row in best_by_size.iterrows():
    current_oracle = float(row["oracle_mae"])
    if previous_row is None:
        marginal_absolute = 0.0
        marginal_percentage = 0.0
        from_size = None
    else:
        previous_oracle = float(previous_row["oracle_mae"])
        marginal_absolute = previous_oracle - current_oracle
        marginal_percentage = marginal_absolute / previous_oracle * 100.0 if previous_oracle else 0.0
        from_size = int(previous_row["number_of_experts"])
    marginal_rows.append(
        {
            "from_number_of_experts": from_size,
            "to_number_of_experts": int(row["number_of_experts"]),
            "best_combination_at_size": row["combination"],
            "oracle_mae": current_oracle,
            "marginal_absolute_improvement": marginal_absolute,
            "marginal_percentage_improvement": marginal_percentage,
        }
    )
    previous_row = row

marginal_improvement_df = pd.DataFrame(marginal_rows)
display(marginal_improvement_df)

print("Marginal oracle improvements")
for _, row in marginal_improvement_df.dropna(subset=["from_number_of_experts"]).iterrows():
    print(
        f"  {int(row['from_number_of_experts'])} -> {int(row['to_number_of_experts'])} experts: "
        f"{row['marginal_absolute_improvement']:.6f} MAE "
        f"({row['marginal_percentage_improvement']:.2f}%)"
    )

## Final Output

`all_combinations_df` is displayed again at the end so the full sorted result table is the final visible notebook output.

In [ ]:
display(all_combinations_df)